# CRS Data Loader v2

Simple 6-phase workflow for loading CRS master data:
1. **Tenant & Branding** - Organization setup and UI customization
2. **Boundaries** - Administrative hierarchy (State → District → Block)
3. **Common Masters** - Departments, designations, complaint types
4. **Employees** - Staff accounts with roles and assignments
5. **Localizations** (Optional) - Bulk load translations for new languages
6. **Workflow** - Complaint state machine (APPLY → ASSIGN → RESOLVE)

---

In [1]:
import warnings
import pandas as pd
import requests
from openpyxl import load_workbook
from dataloader.crs_loader import CRSLoader
from pathlib import Path


In [2]:
# Configuration - Edit these values
URL = "http://kong:8000"
USERNAME = "ADMIN"
PASSWORD = "eGov@123"
TENANT_ID = "pg"  # Root tenant for login
TARGET_TENANT = "ka.blr"  # Target tenant for data (can be child like "pg.citya")

loader = CRSLoader(URL)
loader.login(username=USERNAME, password=PASSWORD, tenant_id=TENANT_ID)

# Create target tenant if it doesn't exist (also enables PGR & HRMS modules)
loader.create_tenant(TARGET_TENANT, users=[
      {"username": "ADMIN", "password": "eGov@123", "name": "Admin",
       "roles": ["SUPERUSER", "EMPLOYEE", "CSR", "GRO", "DGRO", "LME"]}
  ])

loader.login(username=USERNAME, password=PASSWORD, tenant_id=TARGET_TENANT)

# Resolve template directory reliably regardless of notebook launch directory
if (Path.cwd() / "dataloader" / "templates").exists():
    TEMPLATE_DIR = Path.cwd() / "dataloader" / "templates"
elif (Path.cwd() / "templates").exists():
    TEMPLATE_DIR = Path.cwd() / "templates"
else:
    raise FileNotFoundError("Could not locate templates directory. Expected ./templates or ./dataloader/templates")


✅ Authentication successful!
   User: ADMIN
   Name: System Administrator
   Tenant: pg
   Token: 90a705cf-c22e-4240-a...
📝 Creating tenant 'ka.blr'...

🔧 Bootstrapping new tenant root 'ka' from 'pg'...
   📋 Schemas: 32 copied, 0 already existed, 0 failed (of 32 total)

[UPLOADING] common-masters.IdFormat
   Tenant: ka
   Records: 178
   API URL: http://kong:8000/mdms-v2/v2/_create/common-masters.IdFormat
   [OK] [1/178] 1
   [OK] [2/178] 2
   [OK] [3/178] 3
   [OK] [4/178] 4
   [OK] [5/178] 5
   [OK] [6/178] 6
   [OK] [7/178] 7
   [OK] [8/178] 8
   [OK] [9/178] 9
   [OK] [10/178] 10
   [OK] [11/178] 11
   [OK] [12/178] 12
   [OK] [13/178] 13
   [OK] [14/178] 14
   [OK] [15/178] 15
   [OK] [16/178] 16
   [OK] [17/178] 17
   [OK] [18/178] 18
   [OK] [19/178] 19
   [OK] [20/178] 20
   [OK] [21/178] 21
   [OK] [22/178] 22
   [OK] [23/178] 23
   [OK] [24/178] 24
   [OK] [25/178] 25
   [OK] [26/178] 26
   [OK] [27/178] 27
   [OK] [28/178] 28
   [OK] [29/178] 29
   [OK] [30/178] 30
   [OK] [

In [3]:
# Phase 2a - Generate Boundary Template
# Creates hierarchy definition and downloads an Excel template to fill

BOUNDARY_HIERARCHY = "ADMIN"  # or "ADMIN" 
BOUNDARY_LEVELS = ["State","District","Locality"]  # Top to bottom

template_path = loader.load_hierarchy(
    name=BOUNDARY_HIERARCHY,
    levels=BOUNDARY_LEVELS,
    target_tenant=TARGET_TENANT,
    output_dir="upload"
)
print(f"Template saved to: {template_path}")
# Fill the template with your boundary data, then run Phase 2b below


PHASE 2a: BOUNDARY HIERARCHY & TEMPLATE
Tenant: ka.blr
Hierarchy: ADMIN
Levels: State -> District -> Locality

[1/4] Building hierarchy definition...
   Created 3 level definitions

[2/4] Creating hierarchy...

✅ [SUCCESS] Boundary hierarchy created
   Tenant: ka.blr
   Hierarchy Type: ADMIN
   Levels: 3
   Hierarchy created successfully

[3/4] Generating template...

✅ Template generation initiated
   Task ID: 402c4a0a-d784-4328-8642-049f3658fb9b
   Status: inprogress

[4/4] Waiting for template...

⏳ Polling for template generation (max 30 attempts)...
   Attempt 1/30: Status = inprogress
   Attempt 2/30: Status = failed

❌ Template generation failed
   Error: No error details provided
   Additional Details: {
  "error": {
    "code": "FETCHING_COLUMN_ERROR",
    "status": 400,
    "message": "Error fetching Column Headers From Schema",
    "description": "Error fetching column Headers From Schema (either boundary code column not found or given  Campaign Type not found in schema) Ch

In [4]:
# Phase 2b - Load Boundaries from filled template
# Use the template path from Phase 2a, or specify your own filled boundary file

BOUNDARY_FILE = str(TEMPLATE_DIR / "Boundary_Master.xlsx")  # Or: "upload/boundary_filled_statea_REVENUE.xlsx"

loader.load_boundaries(BOUNDARY_FILE, target_tenant=TARGET_TENANT, hierarchy_type=BOUNDARY_HIERARCHY)



PHASE 2: BOUNDARIES
File: Boundary_Master.xlsx
Hierarchy: ADMIN

[1/2] Uploading boundary file...

📤 Uploading file: Boundary_Master.xlsx
✅ File uploaded successfully!
   FileStore ID: dd6c9758-89d6-4f0c-ac18-ba2e6d470576
   FileStore ID: dd6c9758-89d6-4f0c-ac18-ba2e6d470576

[2/2] Processing boundary data...

📖 Reading boundary data from: /home/jovyan/work/dataloader/dataloader/templates/Boundary_Master.xlsx
   Found 6 boundary records
   Columns: ['code', 'name', 'boundaryType', 'parentCode']
   Hierarchy: State → District → Locality
   Using standard format (code, boundaryType, parentCode)
   Boundary types match hierarchy - no mapping needed
   ✅ Created boundary: KA
   ✅ Created relationship: KA [State] (root)
   ✅ Created boundary: KA_DISTONE
   ✅ Created relationship: KA_DISTONE [District] (parent: KA)
   ✅ Created boundary: KA_LOCONE
   ✅ Created relationship: KA_LOCONE [Locality] (parent: KA_DISTONE)
   ✅ Created boundary: KA_LOCTWO
   ✅ Created relationship: KA_LOCTWO [Local

{'status': 'completed',
 'boundaries_created': 6,
 'relationships_created': 6,
 'errors': []}

In [5]:
# Phase 3 - Common Masters
loader.load_common_masters(str(TEMPLATE_DIR / "Common and Complaint Master.xlsx"), target_tenant=TARGET_TENANT)



PHASE 3: COMMON MASTERS
File: Common and Complaint Master.xlsx

[1/2] Loading departments & designations...
📥 Fetching departments from MDMS for tenant: ka.blr
   ✅ Found 13 department(s)
📥 Fetching designations from MDMS for tenant: ka.blr
   ✅ Found 29 designation(s)
📥 Fetching departments from MDMS for tenant: ka
   ✅ Found 13 department(s)
📥 Fetching designations from MDMS for tenant: ka
   ✅ Found 29 designation(s)
   Existing data on ka.blr: 13 dept(s), 29 desig(s)
   Existing data on ka: 13 dept(s), 29 desig(s)
   Departments: 0 existing (reused), 2 new (to create)
   Designations: 0 existing (reused), 2 new (to create)
   Creating 2 departments...

[UPLOADING] common-masters.Department
   Tenant: ka.blr
   Records: 2
   API URL: http://kong:8000/mdms-v2/v2/_create/common-masters.Department
   [OK] [1/2] DEPT_36
   [OK] [2/2] DEPT_37
[SUMMARY] Created: 2
[SUMMARY] Already Exists: 0
[SUMMARY] Failed: 0

📝 Updating Excel file: /home/jovyan/work/dataloader/dataloader/templates/Com

{'departments': {'created': 2, 'exists': 0, 'failed': 0, 'errors': []},
 'designations': {'created': 2, 'exists': 0, 'failed': 0, 'errors': []},
 'complaint_types': {'created': 0,
  'exists': 0,
  'failed': 1,
  'errors': [{'id': 'TabBroken',
    'error': 'extraneous key [menuPathName] is not permitted'}]}}

In [ ]:
# Phase 4 - Employees
loader.load_employees(str(TEMPLATE_DIR / "Employee_Master_Dynamic_statea.xlsx"), target_tenant=TARGET_TENANT)


---

## Phase 5: Localizations (Optional)

Load bulk localization messages for a new language. Use this when you need to add Hindi, Tamil, Punjabi, or other regional languages.

**Prerequisites:**
- You need a localization Excel file with a `Localization` sheet
- Required columns: `Code`, `Message`, `Locale` (e.g., `hi_IN`, `pa_IN`)
- Optional columns: `Module`

**To add a new language:**
1. Get the localization template from `templates/localization.xlsx`
2. Fill in translations for the new locale
3. Run the cell below with `language_label` and `locale_code` to enable the language in the UI dropdown

In [ ]:
# Phase 5 - Localizations (Optional)
# For bulk translations - upload localization Excel with translated messages

# Option A: Just upload messages (no new language in dropdown)
loader.load_localizations(str(TEMPLATE_DIR / "localization.xlsx"), target_tenant=TARGET_TENANT)

# Option B: Upload messages AND enable new language in UI dropdown
# Uncomment and edit the values below:

# loader.load_localizations(
#     str(TEMPLATE_DIR / "localization.xlsx"),
#     target_tenant=TARGET_TENANT,
#     language_label="ਪੰਜਾਬੀ",   # Display name in UI (e.g., "Hindi", "ਪੰਜਾਬੀ", "తెలుగు")
#     locale_code="pa_IN"         # Locale code (e.g., "hi_IN", "pa_IN", "te_IN")
# )


---

## Phase 6: Workflow Configuration

Configure the PGR complaint workflow state machine from a JSON file. This defines the complaint lifecycle:
- **APPLY** → Citizen/CSR creates complaint
- **PENDINGFORASSIGNMENT** → GRO assigns to field worker  
- **PENDINGATLME** → Field worker resolves or reassigns
- **RESOLVED/REJECTED** → Terminal states with rating option

**Template:** Use `../default-data-handler/src/main/resources/PgrWorkflowConfig.json` as a starting point.

The JSON file should have a `BusinessServices` array. Use `{tenantid}` as a placeholder - it will be replaced with your target tenant.

In [ ]:
# Phase 6 - Workflow
# Loads the PGR complaint workflow state machine from JSON file

# Default PGR workflow template (copy and modify as needed)
WORKFLOW_JSON = str(TEMPLATE_DIR / "PgrWorkflowConfig.json")

loader.load_workflow(WORKFLOW_JSON, target_tenant=TARGET_TENANT)


---

## Rollback / Delete Data

Use these cells to undo data loading. Run only what you need to rollback.

In [ ]:
# Rollback Phase 2 - Delete Boundaries
# Deletes all boundary entities and relationships for the tenant
loader.delete_boundaries(TARGET_TENANT)

In [ ]:
# Rollback Phase 3 - Delete Common Masters
# Deletes departments, designations, and complaint types
loader.rollback_common_masters(TARGET_TENANT)

In [ ]:
# Rollback Phase 1 - Delete Tenant Config
# Deletes tenant and branding configuration
loader.rollback_tenant(TARGET_TENANT)

In [ ]:
# Delete specific MDMS schema data (granular control)
loader.delete_mdms("common-masters.Department", TARGET_TENANT)
loader.delete_mdms("common-masters.Designation", TARGET_TENANT)
loader.delete_mdms("RAINMAKER-PGR.ServiceDefs", TARGET_TENANT)

In [ ]:
# FULL RESET - Delete everything (MDMS + Boundaries)
# WARNING: This deletes ALL data for the tenant!
loader.full_reset("REVENUE", TARGET_TENANT)

**Note:** Employees (Phase 4) cannot be deleted via API - they are managed in HRMS and must be deactivated manually if needed.